# BERT-base Fine-Tuning for YouTube Educational Content Classification

This notebook fine-tunes a BERT-base model to classify YouTube transcripts as educational or non-educational.

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- Dataset files: `train_dataset.json`, `val_dataset.json`, `dataset_metadata.json`

**Process:**
1. Setup environment and install dependencies
2. Load and prepare datasets
3. Initialize BERT model with special tokens
4. Train model
5. Evaluate and save model

## Cell 1: Install Dependencies

In [ ]:
# Install required packages
!pip install transformers datasets torch scikit-learn pandas -q

## Cell 2: Mount Google Drive (for model persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")
print("Models will be saved to: /content/drive/MyDrive/bert_edu_classifier")

## Cell 3: Upload Dataset Files

Upload these 3 files using the file upload button:
- `train_dataset.json`
- `val_dataset.json`
- `dataset_metadata.json`

In [ ]:
from google.colab import files

print("Please upload these 3 files:")
print("  1. train_dataset.json")
print("  2. val_dataset.json")
print("  3. dataset_metadata.json")
print()

uploaded = files.upload()

print(f"\n✓ Uploaded {len(uploaded)} file(s)")

## Cell 4: Load and Prepare Data

In [ ]:
import json
import torch
from datasets import Dataset, DatasetDict
import pandas as pd

# Load datasets
with open('train_dataset.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open('val_dataset.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

with open('dataset_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print("Dataset loaded successfully!")
print("="*80)
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nLabel distribution:")
print(f"  Train - Educational: {metadata['label_distribution']['train']['educational']}")
print(f"  Train - Non-educational: {metadata['label_distribution']['train']['non_educational']}")
print(f"  Val - Educational: {metadata['label_distribution']['validation']['educational']}")
print(f"  Val - Non-educational: {metadata['label_distribution']['validation']['non_educational']}")
print(f"\nSpecial tokens: {', '.join(metadata['special_tokens'])}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict({
    'text': [item['text'] for item in train_data],
    'label': [item['label'] for item in train_data]
})

val_dataset = Dataset.from_dict({
    'text': [item['text'] for item in val_data],
    'label': [item['label'] for item in val_data]
})

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

print(f"\n✓ HuggingFace datasets created")
print(dataset_dict)

## Cell 5: Initialize Tokenizer and Add Special Tokens

In [ ]:
from transformers import AutoTokenizer

# Load BERT tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Original vocabulary size: {len(tokenizer)}")

# Add special tokens
special_tokens = metadata['special_tokens']
special_tokens_dict = {'additional_special_tokens': special_tokens}
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)

print(f"\n✓ Added {num_added_tokens} special tokens:")
for token in special_tokens:
    print(f"  - {token}")
print(f"\n✓ New vocabulary size: {len(tokenizer)}")

## Cell 6: Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize text with padding and truncation."""
    return tokenizer(
        examples['text'],
        padding='max_length',  # Pad to max length
        truncation=True,       # Truncate if longer than 512 tokens
        max_length=512
    )

print("Tokenizing datasets...")

tokenized_datasets = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print("\n✓ Tokenization complete!")
print(f"\nSample tokenized input:")
print(f"  Keys: {tokenized_datasets['train'][0].keys()}")
print(f"  Input IDs shape: {len(tokenized_datasets['train'][0]['input_ids'])}")

## Cell 7: Initialize Model

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load BERT model for binary classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification
    id2label={0: 'non_educational', 1: 'educational'},
    label2id={'non_educational': 0, 'educational': 1}
)

# Resize model embeddings to accommodate new special tokens
model.resize_token_embeddings(len(tokenizer))

print(f"✓ Model initialized: {model_name}")
print(f"✓ Model embeddings resized to: {len(tokenizer)}")
print(f"✓ Total parameters: {model.num_parameters():,}")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Cell 8: Define Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    
    # Training hyperparameters
    num_train_epochs=4,              # 4 epochs for fine-tuning
    per_device_train_batch_size=8,   # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    learning_rate=2e-5,              # Standard for BERT fine-tuning
    weight_decay=0.01,               # Regularization
    warmup_steps=100,                # Warmup for 100 steps
    
    # Evaluation and saving
    eval_strategy='epoch',           # Evaluate after each epoch
    save_strategy='epoch',           # Save checkpoint after each epoch
    load_best_model_at_end=True,     # Load best model at end
    metric_for_best_model='f1',      # Use F1 score to select best model
    
    # Logging
    logging_dir='./logs',
    logging_steps=10,
    report_to='none',                # Disable wandb/tensorboard
    
    # Hardware optimization
    fp16=True,                       # Use mixed precision for faster training
    
    # Reproducibility
    seed=42
)

print("Training configuration:")
print("="*80)
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Eval batch size: {training_args.per_device_eval_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Weight decay: {training_args.weight_decay}")
print(f"  Warmup steps: {training_args.warmup_steps}")
print(f"  Mixed precision (FP16): {training_args.fp16}")
print(f"  Evaluation strategy: {training_args.eval_strategy}")
print(f"  Best model metric: {training_args.metric_for_best_model}")

## Cell 9: Define Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def compute_metrics(eval_pred):
    """Compute metrics for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    
    # Confusion matrix
    cm = confusion_matrix(labels, predictions)
    
    print("\nConfusion Matrix:")
    print("                 Predicted")
    print("                 Non-Edu  Educational")
    print(f"Actual Non-Edu     {cm[0][0]:3d}      {cm[0][1]:3d}")
    print(f"Actual Educational {cm[1][0]:3d}      {cm[1][1]:3d}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("✓ Metrics function defined")

## Cell 10: Initialize Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("✓ Trainer initialized")
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Cell 11: Train Model

This will take approximately 15-30 minutes depending on dataset size and GPU.

In [ ]:
print("Starting training...")
print("="*80)
print(f"Training {len(tokenized_datasets['train'])} samples")
print(f"Validating on {len(tokenized_datasets['validation'])} samples")
print(f"Total epochs: {training_args.num_train_epochs}")
print("="*80)

# Train
train_result = trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"Final training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Samples per second: {train_result.metrics['train_samples_per_second']:.2f}")

## Cell 12: Evaluate Model

In [ ]:
print("Evaluating on validation set...")
print("="*80)

eval_results = trainer.evaluate()

print("\nVALIDATION RESULTS:")
print("="*80)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print("\nMetrics Summary:")
print(f"  Accuracy:  {eval_results['eval_accuracy']:.2%}")
print(f"  Precision: {eval_results['eval_precision']:.2%}")
print(f"  Recall:    {eval_results['eval_recall']:.2%}")
print(f"  F1 Score:  {eval_results['eval_f1']:.2%}")

## Cell 13: Save Model

In [ ]:
# Save to Google Drive for persistence
drive_save_path = '/content/drive/MyDrive/bert_edu_classifier'
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"✓ Model saved to Google Drive: {drive_save_path}")

# Also save locally in Colab session
local_save_path = './fine_tuned_model'
model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)

print(f"✓ Model saved locally: {local_save_path}")
print("\n" + "="*80)
print("Model files saved successfully!")
print("="*80)

## Cell 14: Test Predictions

In [ ]:
def predict_educational(text):
    """Helper function to predict if text is educational."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    
    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probabilities = torch.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()
    
    return {
        'prediction': 'educational' if prediction == 1 else 'non_educational',
        'confidence': confidence,
        'probabilities': {
            'non_educational': probabilities[0][0].item(),
            'educational': probabilities[0][1].item()
        }
    }

# Test on sample texts
test_samples = [
    "Today we're going to learn about machine learning algorithms and neural networks. First, let's understand what a neuron is.",
    "Don't forget to like and subscribe! Check out my merch link in the description below!",
    "The derivative of x squared is 2x, which we can prove using the limit definition of the derivative.",
    "This gameplay is insane! Watch me get this victory royale. Smash that like button!"
]

print("Test Predictions:")
print("="*80)

for idx, sample in enumerate(test_samples, 1):
    result = predict_educational(sample)
    print(f"\nSample {idx}:")
    print(f"  Text: {sample[:80]}...")
    print(f"  Prediction: {result['prediction']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print(f"  Probabilities:")
    print(f"    Educational: {result['probabilities']['educational']:.2%}")
    print(f"    Non-educational: {result['probabilities']['non_educational']:.2%}")

## Cell 15: Download Model Files

Create a ZIP file for easy download and transfer to your local machine.

In [ ]:
# Create ZIP file
!zip -r fine_tuned_model.zip ./fine_tuned_model

print("✓ Model files zipped")
print("\nDownloading model...")

# Download
from google.colab import files
files.download('fine_tuned_model.zip')

print("\n" + "="*80)
print("SUCCESS!")
print("="*80)
print("Your fine-tuned model has been downloaded as 'fine_tuned_model.zip'")
print("\nNext steps:")
print("  1. Extract the ZIP file on your local machine")
print("  2. Run 'python use_finetuned_model.py' to test inference")
print("  3. Run 'python evaluate_models.py' to compare models")
print("  4. Update 'program.py' to use your fine-tuned model")